# bert-gg-qa-testing

Submit-only notebook for Google QUEST Q&A Labeling.
Loads saved artifacts from the training notebook and writes submission CSV.


In [ ]:
import os, json
import numpy as np
import pandas as pd

out_dir = '/kaggle/working/ggqa_bert_models/artifacts'
sample = pd.read_csv('/kaggle/input/competitions/google-quest-challenge/sample_submission.csv')


In [ ]:
with open(f'{out_dir}/target_cols.json') as f:
    target_cols = json.load(f)
with open(f'{out_dir}/label_values.json') as f:
    label_values = json.load(f)
with open(f'{out_dir}/clippings.json') as f:
    CLIPPINGS = json.load(f)

ensemble_test = np.load(f'{out_dir}/test_ensemble.npy')


In [ ]:
def snap_to_known_labels(preds, target_cols, label_values):
    out = np.zeros_like(preds)
    for j, c in enumerate(target_cols):
        vals = np.array(label_values[c], dtype=np.float32)
        idx = np.abs(preds[:, [j]] - vals.reshape(1, -1)).argmin(axis=1)
        out[:, j] = vals[idx]
    return out

def apply_clipping(preds, target_cols):
    out = preds.copy()
    for j, c in enumerate(target_cols):
        if c in CLIPPINGS:
            lo, hi = CLIPPINGS[c]
            out[:, j] = np.clip(out[:, j], lo, hi)
    return out


In [ ]:
final_test = snap_to_known_labels(apply_clipping(ensemble_test, target_cols), target_cols, label_values)
sub = sample.copy()
sub[target_cols] = final_test
sub.to_csv(f'{out_dir}/submission.csv', index=False)
sub.head()
